# Observabilidad · Vistas de salud del pipeline

## Propósito

Crear las vistas que alimentan el dashboard de salud del pipeline. Resumen en un solo lugar de cómo corrió la última ejecución, cómo evolucionan las ejecuciones en el tiempo y qué errores se han presentado.

## Flujo

1. Crear la vista de la última ejecución, con una fila por tarea.
2. Crear la vista histórica, con una fila por ejecución.
3. Crear la vista de errores de los últimos 30 días.

## Origen

Registro de ejecución del pipeline (`etl_log`), donde cada tarea deja su inicio, fin, estado, filas procesadas y mensaje de error.

## Destino

Esquema de control del proyecto.

| Vista | Grano | Qué responde |
|---|---|---|
| `vw_salud_ultima_ejecucion` | una fila por tarea de la última ejecución | ¿Cómo corrió la última vez? ¿Qué tarea tardó más? |
| `vw_salud_historico` | una fila por ejecución | ¿Cómo evolucionan la duración, el éxito y el volumen? |
| `vw_salud_errores` | una fila por error de los últimos 30 días | ¿Qué falló, dónde y cuándo? |

## Reglas de las vistas

* La última ejecución se identifica por la fecha de inicio más reciente.
* Las capas se normalizan en cinco grupos: bronze, silver, gold, metadata y control.
* La duración de cada tarea se calcula en segundos (fin menos inicio).
* El histórico considera solo las ejecuciones desde el 2026-10-08, cuando el registro de filas procesadas quedó completo y validado.
* Una ejecución se marca como exitosa solo si todas sus tareas terminaron en EXITOSO.

## Cómo leer las filas procesadas

El significado depende de la capa:

* **Bronze:** filas del corte que entró en esa ejecución.
* **Silver y Gold:** total de filas que quedó en la tabla destino al terminar la tarea.
* Las tareas de control y los logs de error no cargan tablas, por eso quedan sin valor.

Las cifras de Bronze y de Silver o Gold no son comparables entre sí.

## Resultado esperado

Tres vistas listas para usar como fuentes de datos del dashboard de salud del pipeline.

In [0]:
%sql
CREATE OR REPLACE VIEW metadata_dataco.supply_chain.vw_salud_ultima_ejecucion AS
WITH ultima AS (
  SELECT id_ejecucion
  FROM metadata_dataco.supply_chain.hd_etl_log
  GROUP BY id_ejecucion
  ORDER BY MAX(fecha_inicio) DESC
  LIMIT 1
)
SELECT
  l.id_ejecucion,
  l.notebook,
  CASE WHEN l.capa IN ('metadata', 'control') THEN 'control' ELSE l.capa END AS capa,
  l.estado,
  l.fecha_inicio,
  l.fecha_fin,
  timestampdiff(SECOND, l.fecha_inicio, l.fecha_fin) AS duracion_seg,
  l.filas_procesadas,
  l.mensaje_error
FROM metadata_dataco.supply_chain.hd_etl_log l
JOIN ultima u ON l.id_ejecucion = u.id_ejecucion;

In [0]:
%sql
CREATE OR REPLACE VIEW metadata_dataco.supply_chain.vw_salud_historico AS
SELECT
  id_ejecucion,
  MIN(fecha_inicio) AS fecha_inicio,
  MAX(fecha_fin) AS fecha_fin,
  timestampdiff(SECOND, MIN(fecha_inicio), MAX(fecha_fin)) AS duracion_total_seg,
  COUNT(*) AS tasks_total,
  SUM(CASE WHEN estado = 'EXITOSO' THEN 1 ELSE 0 END) AS tasks_exitosas,
  SUM(CASE WHEN estado = 'ERROR' THEN 1 ELSE 0 END) AS tasks_error,
  SUM(CASE WHEN estado = 'EN_PROCESO' THEN 1 ELSE 0 END) AS tasks_en_proceso,
  CASE
    WHEN SUM(CASE WHEN estado = 'ERROR' THEN 1 ELSE 0 END) > 0 THEN 'ERROR'
    WHEN SUM(CASE WHEN estado = 'EN_PROCESO' THEN 1 ELSE 0 END) > 0 THEN 'EN_PROCESO'
    ELSE 'EXITOSO'
  END AS estado_global,
  SUM(CASE WHEN capa = 'bronze' THEN filas_procesadas END) AS filas_bronze,
  SUM(CASE WHEN capa = 'silver' THEN filas_procesadas END) AS filas_silver,
  SUM(CASE WHEN capa = 'gold' THEN filas_procesadas END) AS filas_gold
FROM metadata_dataco.supply_chain.hd_etl_log
WHERE fecha_inicio >= TIMESTAMP '2026-10-08'
GROUP BY id_ejecucion;

In [0]:
%sql
CREATE OR REPLACE VIEW metadata_dataco.supply_chain.vw_salud_errores AS
SELECT
  fecha_inicio,
  id_ejecucion,
  notebook,
  CASE WHEN capa IN ('metadata', 'control') THEN 'control' ELSE capa END AS capa,
  LEFT(mensaje_error, 300) AS mensaje_corto,
  mensaje_error
FROM metadata_dataco.supply_chain.hd_etl_log
WHERE estado = 'ERROR'
  AND fecha_inicio >= current_timestamp() - INTERVAL 30 DAYS
ORDER BY fecha_inicio DESC;

In [0]:
%sql
SELECT notebook, capa, estado, duracion_seg, filas_procesadas
FROM metadata_dataco.supply_chain.vw_salud_ultima_ejecucion
ORDER BY fecha_inicio;

In [0]:
%sql
SELECT notebook, capa, estado, duracion_seg, filas_procesadas
FROM metadata_dataco.supply_chain.vw_salud_ultima_ejecucion
ORDER BY fecha_inicio;